# MOD16A2 Aggregation Debug Notebook

Reproduces the aggregation pipeline for a single year + small batch of HRUs so errors
can be isolated and inspected interactively.  Mirrors the logic in
`src/nhf_spatial_targets/aggregate/_driver.py`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import xarray as xr

# ── project / datastore paths ──────────────────────────────────────────────
PROJECT_DIR  = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
DATASTORE    = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
FABRIC_PATH  = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2_param_v2/gfv2/fabric/gfv2_nhru_merged.gpkg")
ID_COL       = "nat_hru_id"

# ── choose one consolidated year for debugging ─────────────────────────────
DEBUG_YEAR   = 2001
NC_PATH      = DATASTORE / "mod16a2_v061" / f"mod16a2_v061_{DEBUG_YEAR}_consolidated.nc"
BATCH_SIZE   = 5000

print(f"NC path:  {NC_PATH}  exists={NC_PATH.exists()}")
print(f"Fabric:   {FABRIC_PATH}  exists={FABRIC_PATH.exists()}")


## 1  Inspect the consolidated source NC

In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print("\n── coordinate attrs ──")
for name in ds.coords:
    print(f"  {name}: axis={ds[name].attrs.get('axis')!r}  "
          f"standard_name={ds[name].attrs.get('standard_name')!r}  "
          f"shape={ds[name].shape}")


In [ ]:
# Check that detect_coords resolves the right coord names
import sys
sys.path.insert(0, str(Path("../src")))

from nhf_spatial_targets.aggregate._coords import detect_coords

x, y, t = detect_coords(ds, "ET_500m")
print(f"Detected:  x={x!r}  y={y!r}  t={t!r}")


## 2  Load the fabric and create a small test batch

In [ ]:
from nhf_spatial_targets.aggregate._driver import load_and_batch_fabric

fabric_batched = load_and_batch_fabric(FABRIC_PATH, batch_size=BATCH_SIZE)
print(f"Fabric rows: {len(fabric_batched)}  batches: {fabric_batched['batch_id'].nunique()}")
print(f"CRS: {fabric_batched.crs}")
fabric_batched.head(3)


In [ ]:
# Grab a single small batch
TEST_BATCH_ID = 0
batch_gdf = fabric_batched[fabric_batched["batch_id"] == TEST_BATCH_ID].drop(columns=["batch_id"])
print(f"Batch {TEST_BATCH_ID}: {len(batch_gdf)} HRUs")
print(f"Bounds: {batch_gdf.total_bounds}")
batch_gdf.head(3)


## 3  Diagnose the CRS mismatch, then reproduce compute_or_load_weights

`consolidate_mod16a2` reprojects the raw sinusoidal HDF tiles to **WGS84 (EPSG:4326)**
and renames the spatial coords to `lon` / `lat`.  If `UserCatData` is given
`proj_ds=MODIS_SINUSOIDAL_PROJ` (metres), it reprojects the fabric bounds into
sinusoidal metres and tries to slice the `lon` coordinate (±180 °) with those
large metre values → empty spatial subset → `Length of values (0)` downstream.

In [ ]:
from gdptools import UserCatData, WeightGen

from nhf_spatial_targets.aggregate.mod16a2 import ADAPTER, MODIS_SINUSOIDAL_PROJ
from nhf_spatial_targets.aggregate._driver import WEIGHT_GEN_CRS

PERIOD = (f"{DEBUG_YEAR}-01-01", f"{DEBUG_YEAR}-12-31")

# ── Show what goes wrong with the sinusoidal CRS ───────────────────────────
from gdptools.utils import _get_shp_bounds_w_buffer

bad_bounds = _get_shp_bounds_w_buffer(
    gdf=batch_gdf, ds=ds, crs=MODIS_SINUSOIDAL_PROJ, lon=x, lat=y
)
print(f"Bounds with sinusoidal CRS (metres): {bad_bounds}")
good_bounds = _get_shp_bounds_w_buffer(
    gdf=batch_gdf, ds=ds, crs="EPSG:4326", lon=x, lat=y
)
print(f"Bounds with EPSG:4326      (degrees): {good_bounds}")
print(f"lon coord range: {float(ds[x].min()):.2f} … {float(ds[x].max()):.2f}")


In [ ]:
# Create UserCatData with the CORRECT source CRS (EPSG:4326)
user_data_w = UserCatData(
    ds=ds,
    proj_ds="EPSG:4326",   # consolidated NC is in WGS84, not sinusoidal
    x_coord=x,
    y_coord=y,
    t_coord=t,
    var=[ADAPTER.variables[0]],
    f_feature=batch_gdf,
    proj_feature=WEIGHT_GEN_CRS,
    id_feature=ID_COL,
    period=list(PERIOD),
)
print("UserCatData created OK")
print(f"source_time_period: {user_data_w.source_time_period}")

# Check the subset shape — should be (n_times, n_lat, n_lon), not (n_times, 0, 0)
da_subset = user_data_w.get_source_subset(ADAPTER.variables[0])
print(f"Source subset shape: {da_subset.shape}")
print(f"Time range: {da_subset.time.values[[0, -1]]}")
print(f"NaN fraction: {float(da_subset.isnull().mean()):.3f}")


In [ ]:
ds.ET_500m.values


In [ ]:
wg = WeightGen(
    user_data=user_data_w,
    method="serial",
    weight_gen_crs=WEIGHT_GEN_CRS,
)
weights = wg.calculate_weights()
print(f"Weights shape: {weights.shape}")
print(f"Weights columns: {list(weights.columns)}")
print(f"Unique HRUs in weights: {weights[ID_COL].nunique()}  (batch has {len(batch_gdf)})")
weights.head()


In [ ]:
# Which batch HRUs got NO weights?
hrus_with_weights = set(weights[ID_COL].astype(str).unique())
hrus_in_batch    = set(batch_gdf[ID_COL].astype(str).unique())
missing = hrus_in_batch - hrus_with_weights
print(f"HRUs with no grid-cell intersection: {len(missing)} → {sorted(missing)[:10]}")


## 4  Reproduce aggregate_variables_for_batch

In [ ]:
from gdptools import AggGen, UserCatData

user_data_a = UserCatData(
    ds=ds,
    proj_ds="EPSG:4326",   # consolidated NC is in WGS84, not sinusoidal
    x_coord=x,
    y_coord=y,
    t_coord=t,
    var=[ADAPTER.variables[0]],
    f_feature=batch_gdf,
    proj_feature=WEIGHT_GEN_CRS,
    id_feature=ID_COL,
    period=list(PERIOD),
)

agg = AggGen(
    user_data=user_data_a,
    stat_method="mean",
    agg_engine="serial",
    agg_writer="none",
    weights=weights,
)
gdf, batch_ds = agg.calculate_agg()
print(batch_ds)
print(f"\nResult dims: {dict(batch_ds.dims)}")


In [ ]:
# Spot-check one HRU time series
import matplotlib.pyplot as plt

hru_id = batch_gdf[ID_COL].iloc[0]
ts = batch_ds["ET_500m"].sel({ID_COL: hru_id})
ts.plot(figsize=(12, 3), marker="o", markersize=3)
plt.title(f"MOD16A2 ET_500m — HRU {hru_id} — {DEBUG_YEAR}")
plt.tight_layout()
plt.show()


## 5  Full aggregate_year for one batch (reuse weights)

In [ ]:
from nhf_spatial_targets.aggregate._driver import aggregate_variables_for_batch

# Apply the adapter's pre_aggregate_hook (fill-value masking) to match what
# aggregate_year does internally.  Without this, section 5 would aggregate
# the raw values including MODIS special codes (water, cloud, no-data).
ds_masked = ADAPTER.pre_aggregate_hook(ds) if ADAPTER.pre_aggregate_hook else ds

batch_ds2 = aggregate_variables_for_batch(
    batch_gdf=batch_gdf,
    source_ds=ds_masked,
    variables=list(ADAPTER.variables),
    source_crs=ADAPTER.source_crs,   # "EPSG:4326" — use adapter, not the sinusoidal constant
    x_coord=x,
    y_coord=y,
    time_coord=t,
    id_col=ID_COL,
    weights=weights,
    period=PERIOD,
)
print(batch_ds2)
print(f"\nET_500m range: {float(batch_ds2['ET_500m'].min()):.2f} … {float(batch_ds2['ET_500m'].max()):.2f}")
print(f"ET_500m NaN count: {int(batch_ds2['ET_500m'].isnull().sum())}")


## 6  Run aggregate_year end-to-end (writes per-year intermediate)

In [ ]:
import logging
from pathlib import Path

log_file = Path("../logs/debug_mod16a2_agg.log")
log_file.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename=log_file,
    filemode="w",
    force=True,
)
print(f"Logging to {log_file.resolve()}")


In [ ]:
from nhf_spatial_targets.aggregate._driver import per_year_output_path, weight_cache_path
from nhf_spatial_targets.workspace import load as load_project

project = load_project(PROJECT_DIR)

# ── Delete stale per-year intermediate ────────────────────────────────────
# aggregate_year is idempotent; it skips if the file exists.  Any file from
# a pre-fix run (wrong source_crs) must be removed before re-running.
stale_nc = per_year_output_path(project, ADAPTER.source_key, DEBUG_YEAR)
if stale_nc.exists():
    stale_nc.unlink()
    print(f"Deleted stale intermediate: {stale_nc}")
else:
    print(f"No stale intermediate: {stale_nc}")

# ── Delete stale weight caches ─────────────────────────────────────────────
# Weights were computed with MODIS_SINUSOIDAL_PROJ — all batches are invalid.
weight_dir = project.workdir / "weights"
stale_weights = sorted(weight_dir.glob(f"{ADAPTER.source_key}_batch*.csv"))
for w in stale_weights:
    w.unlink()
print(f"Deleted {len(stale_weights)} stale weight cache(s) from {weight_dir}")


In [ ]:
from nhf_spatial_targets.aggregate._driver import aggregate_year

out_path = aggregate_year(
    adapter=ADAPTER,
    project=project,
    year=DEBUG_YEAR,
    source_file=NC_PATH,
    fabric_batched=fabric_batched,
    id_col=ID_COL,
)
print(f"Wrote: {out_path}")

with xr.open_dataset(out_path) as result:
    print(result)


In [ ]:
result.ET_500m.head().values



## 7  Spatial QC — map of annual-mean ET

Join aggregated ET back to the fabric geometry and plot as a choropleth.
A spatially coherent result (smooth gradients, no obvious tiling artefacts)
confirms the aggregation worked correctly across HRU batches.


In [ ]:
import gc
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr

# ── Load aggregated annual mean (values only, no geometry) ─────────────────
with xr.open_dataset(out_path) as result:
    annual_mean = (
        result["ET_500m"]
        .mean(dim="time")
        .to_dataframe()
        .reset_index()[[ID_COL, "ET_500m"]]
    )

print(f"ET_500m (annual mean) range: {annual_mean['ET_500m'].min():.2f} … {annual_mean['ET_500m'].max():.2f}")
print(f"ET_500m NaN count: {annual_mean['ET_500m'].isna().sum()} / {len(annual_mean)}")

# ── Load fabric: id + simplified geometry only ─────────────────────────────
# Simplify to ~1 km tolerance (fabric is EPSG:5070, units = metres).
# This drastically reduces vertex count without affecting visual quality at
# national scale.
fabric_full = gpd.read_file(FABRIC_PATH, columns=[ID_COL, "geometry"])
fabric_full["geometry"] = fabric_full.geometry.simplify(1000, preserve_topology=False)

# ── Join and report coverage ───────────────────────────────────────────────
plot_gdf = fabric_full.merge(annual_mean, on=ID_COL, how="left")
del fabric_full, annual_mean
gc.collect()

n_matched = plot_gdf["ET_500m"].notna().sum()
n_total   = len(plot_gdf)
print(f"HRUs with aggregated values: {n_matched} / {n_total}  "
      f"({100 * n_matched / n_total:.1f}%)")

# ── Plot ───────────────────────────────────────────────────────────────────
# vmax=100 clips the colormap to a physically meaningful range for CONUS
# annual-mean 8-day ET (kg m⁻² 8d⁻¹); fill-value artefacts (>3270) should
# already be masked by the pre_aggregate_hook.
fig, ax = plt.subplots(figsize=(14, 8))
plot_gdf[plot_gdf["ET_500m"].isna()].plot(
    ax=ax, color="lightgrey", linewidth=0,
)
plot_gdf[plot_gdf["ET_500m"].notna()].plot(
    ax=ax,
    column="ET_500m",
    cmap="YlGnBu",
    linewidth=0,
    legend=True,
    vmin=0,
    vmax=100,
    legend_kwds={"label": "Annual mean ET_500m (kg m⁻² 8d⁻¹)", "shrink": 0.6},
    aspect=None,   # skip expensive equal-aspect calculation
)
ax.set_title(f"MOD16A2 v061 — Annual mean ET_500m — {DEBUG_YEAR}", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

del plot_gdf
gc.collect()


In [ ]:
import xarray as xr
import geopandas as gpd

# ── Diagnose join key and geometry ─────────────────────────────────────────
print(f"Inspecting: {out_path}")

with xr.open_dataset(out_path) as result:
    print(result)
    annual_mean_diag = (
        result["ET_500m"]
        .mean(dim="time")
        .to_dataframe()
        .reset_index()[[ID_COL, "ET_500m"]]
    )

fabric_diag = gpd.read_file(FABRIC_PATH, columns=[ID_COL, "geometry"], rows=5)

print("\n=== annual_mean ===")
print(annual_mean_diag.head())
print(f"dtype: {annual_mean_diag[ID_COL].dtype}")
print(f"sample values: {annual_mean_diag[ID_COL].iloc[:3].tolist()}")
print(f"ET_500m range: {annual_mean_diag['ET_500m'].min():.4f} … {annual_mean_diag['ET_500m'].max():.4f}")
print(f"ET_500m NaN count: {annual_mean_diag['ET_500m'].isna().sum()} / {len(annual_mean_diag)}")

print("\n=== fabric (5 rows) ===")
print(fabric_diag[[ID_COL]].head())
print(f"dtype: {fabric_diag[ID_COL].dtype}")
print(f"sample values: {fabric_diag[ID_COL].iloc[:3].tolist()}")

# Check if a test merge would match
test_merge = fabric_diag.merge(annual_mean_diag.head(5), on=ID_COL, how="inner")
print(f"\nTest inner merge rows matched: {len(test_merge)} (expected > 0)")
